# GitLab Projects Export — v5

## Pipeline
| Phase | What it does | DB action |
|---|---|---|
| **Phase 1** | Date-filter → fetch full project object → check `package.json` | DROP/CREATE table, INSERT one row per project |
| **User gate** | Prints summary, asks to continue | — |
| **Phase 2** | Walk file tree → extract every import statement | ALTER TABLE (add import columns), DELETE+INSERT enriched rows |

## Table: `sandbox_prj_smart_insights.gwma_ui_projects`
Phase-1 columns: `project_id, name, path, path_with_namespace, group_path, web_url, description, visibility, archived, created_at, last_activity_at, updated_at, default_branch, forks_count, star_count, open_issues_count, team_name, package_json`  
Phase-2 adds: `library, import_filename, import_file_url, import_statement`

## 1 · Imports & logging

In [ ]:
import getpass
import json
import logging
import re
import sys
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeout
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import gitlab
import psycopg2
import psycopg2.extras
import requests
from requests.adapters import HTTPAdapter

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True,
)
logger = logging.getLogger('gitlab_export')
logging.getLogger('urllib3').setLevel(logging.WARNING)
logging.getLogger('gitlab').setLevel(logging.WARNING)
logger.info('GitLab Projects Export  v5')


## 2 · Configuration

In [ ]:
GITLAB_URL      = 'https://devcloud.ubs.net'
GROUP_PATH      = 'ubs/gwma'

PER_PAGE        = 100
MAX_WORKERS     = 10
FILE_WORKERS    = 5
MAX_CONNECTIONS = 10
GITLAB_TIMEOUT  = (10, 30)
FUTURE_TIMEOUT  = 120
PROGRESS_EVERY  = 5

UPDATED_SINCE     = datetime(2025, 1, 1, tzinfo=timezone.utc)
SOURCE_EXTENSIONS = ('.jsx', '.tsx', '.js', '.ts')

DB_SCHEMA     = 'sandbox_prj_smart_insights'
DB_TABLE      = 'gwma_ui_projects'
DB_FQTABLE    = f'{DB_SCHEMA}.{DB_TABLE}'
DB_BATCH_SIZE = 200

P1_COLUMNS = [
    ('project_id',          'BIGINT'),
    ('name',                'TEXT'),
    ('path',                'TEXT'),
    ('path_with_namespace', 'TEXT'),
    ('group_path',          'TEXT'),
    ('web_url',             'TEXT'),
    ('description',         'TEXT'),
    ('visibility',          'TEXT'),
    ('archived',            'BOOLEAN'),
    ('created_at',          'TIMESTAMP WITH TIME ZONE'),
    ('last_activity_at',    'TIMESTAMP WITH TIME ZONE'),
    ('updated_at',          'TIMESTAMP WITH TIME ZONE'),
    ('default_branch',      'TEXT'),
    ('forks_count',         'INTEGER'),
    ('star_count',          'INTEGER'),
    ('open_issues_count',   'INTEGER'),
    ('team_name',           'TEXT'),
    ('package_json',        'TEXT'),
]
P2_EXTRA_COLUMNS = [
    ('library',           'TEXT'),
    ('import_filename',   'TEXT'),
    ('import_file_url',   'TEXT'),
    ('import_statement',  'TEXT'),
]
P1_COL_NAMES  = [c[0] for c in P1_COLUMNS]
P2_COL_NAMES  = [c[0] for c in P2_EXTRA_COLUMNS]
ALL_COL_NAMES = P1_COL_NAMES + P2_COL_NAMES

ANY_IMPORT_RE = re.compile(
    r'^[ \t]*import'
    r'(?:\s+(?:'
        r'[\w\$_][\w\$_]*(?:\s*,\s*(?:\*\s+as\s+[\w\$_]+|\{[^}]*\}))?'
        r'|\*\s+as\s+[\w\$_][\w\$_]*'
        r'|\{[^}]*\}'
    r')\s+from)?'
    r"\s*['\"]([^'\"]+)['\"]"
    r'\s*;?[^\n]*',
    re.MULTILINE,
)

_stats_lock = threading.Lock()
run_stats: Dict[str, int] = {
    'total_in_group': 0, 'recent_projects': 0, 'skipped_too_old': 0,
    'has_package_json': 0, 'skipped_no_package_json': 0,
    'p1_errors': 0, 'p1_timeouts': 0,
    'source_files_fetched': 0, 'source_files_empty': 0, 'source_files_error': 0,
    'import_rows_total': 0, 'p2_errors': 0, 'p2_timeouts': 0,
}
def _inc(key, n=1):
    with _stats_lock: run_stats[key] += n

_gitlab_sem = threading.Semaphore(MAX_CONNECTIONS)
logger.info('Config ready')


## 3 · GitLab auth

In [ ]:
private_token = getpass.getpass('Enter your GitLab private token: ')
_session = requests.Session()
_adapter = HTTPAdapter(max_retries=1, pool_connections=MAX_CONNECTIONS, pool_maxsize=MAX_CONNECTIONS)
_session.mount('https://', _adapter)
_session.mount('http://',  _adapter)
_orig = _session.request
def _timed(method, url, **kw):
    kw.setdefault('timeout', GITLAB_TIMEOUT)
    return _orig(method, url, **kw)
_session.request = _timed
client = gitlab.Gitlab(GITLAB_URL, private_token=private_token, session=_session)
logger.info('AUTH | GitLab client ready')


## 4 · DB credentials

In [ ]:
db_password = getpass.getpass('Enter Password for DB User: ')
DB_CONFIG = {
    'host'    : 'greenplum-rdsp.zur.swissbank.com',
    'port'    : 5432,
    'dbname'  : 'gprdsp',
    'user'    : 'ds_rdsp_dev',
    'password': db_password,
}
logger.info('DB   | credentials stored  host=%s', DB_CONFIG['host'])


## 5 · Utility helpers

In [ ]:
def _parse_dt(s):
    if not s: return None
    try: return datetime.fromisoformat(s.replace('Z', '+00:00'))
    except: return None

def is_recent(p):
    dt = _parse_dt(getattr(p, 'updated_at', None) or '')
    return True if dt is None else dt >= UPDATED_SINCE

def extract_team_name(url):
    try:
        segs = url.rstrip('/').split('//', 1)[-1].split('/')[1:]
        return segs[-2] if len(segs) >= 2 else ''
    except: return ''

def _raw_file(project, path, ref):
    with _gitlab_sem:
        try: return project.files.raw(file_path=path, ref=ref).decode('utf-8', errors='replace')
        except: return None

def _repo_tree_pages(project, ref):
    items, page = [], 1
    while True:
        with _gitlab_sem:
            try: batch = project.repository_tree(ref=ref, recursive=True, per_page=PER_PAGE, page=page, get_all=False)
            except Exception as e:
                logger.warning('TREE | page=%d err | %s | %s', page, project.path_with_namespace, e)
                break
        if not batch: break
        items.extend(batch)
        if len(batch) < PER_PAGE: break
        page += 1
    return items

def _coerce(v):
    if v is None: return None
    if isinstance(v, float) and v != v: return None
    if isinstance(v, str) and v.strip() == '': return None
    return v

def _derive_library(m):
    s = m.lower()
    r = []
    if s.startswith('@uwr/'): r.append('uwr')
    if s.startswith('@ubs.websdk/'): r.append('websdk')
    return ', '.join(r)

logger.info('Helpers defined')


## 6 · Phase 1 worker — fetch project + check package.json

In [ ]:
def fetch_project_with_pkg(proj_ref: Any) -> Optional[Dict[str, Any]]:
    ns  = proj_ref.path_with_namespace
    ref = getattr(proj_ref, 'default_branch', None) or 'main'
    with _gitlab_sem:
        try: project = client.projects.get(proj_ref.id)
        except Exception as e:
            logger.error('P1 | project.get FAILED | %s | %s', ns, e)
            _inc('p1_errors'); return None
    ref  = project.default_branch or 'main'
    ns_d = project.namespace or {}
    with _gitlab_sem:
        try: project.files.raw(file_path='package.json', ref=ref); has_pkg = True
        except: has_pkg = False
    if not has_pkg:
        _inc('skipped_no_package_json'); return None
    logger.info('P1 | YES pkg.json | %s', project.path_with_namespace)
    _inc('has_package_json')
    return {
        '_project': project, '_ref': ref,
        'project_id'         : int(project.id),
        'name'               : project.name,
        'path'               : project.path,
        'path_with_namespace': project.path_with_namespace,
        'group_path'         : ns_d.get('full_path'),
        'web_url'            : project.web_url,
        'description'        : project.description,
        'visibility'         : project.visibility,
        'archived'           : bool(project.archived),
        'created_at'         : project.created_at,
        'last_activity_at'   : project.last_activity_at,
        'updated_at'         : project.updated_at,
        'default_branch'     : ref,
        'forks_count'        : int(project.forks_count)       if project.forks_count       is not None else None,
        'star_count'         : int(project.star_count)        if project.star_count        is not None else None,
        'open_issues_count'  : int(project.open_issues_count) if project.open_issues_count is not None else None,
        'team_name'          : extract_team_name(project.web_url),
        'package_json'       : 'Yes',
    }

logger.info('fetch_project_with_pkg defined')


## 7 · Phase 2 worker — scan imports

In [ ]:
def scan_imports(p1_row: Dict[str, Any]) -> List[Dict[str, Any]]:
    project = p1_row['_project']
    ref     = p1_row['_ref']
    ns      = project.path_with_namespace
    web_url = project.web_url
    t0      = time.perf_counter()
    logger.info('P2 | START | %s', ns)
    all_items = _repo_tree_pages(project, ref)
    src_items = [i for i in all_items
                 if i.get('type') == 'blob' and i['path'].endswith(SOURCE_EXTENSIONS)]
    logger.info('P2 | tree=%d src=%d | %s', len(all_items), len(src_items), ns)
    base = {k: p1_row[k] for k in P1_COL_NAMES}
    if not src_items:
        return [{**base, 'library':'', 'import_filename':'', 'import_file_url':'', 'import_statement':''}]
    import_rows = []
    lf = le = lerr = ls = 0
    def process_file(item):
        nonlocal lf, le, lerr, ls
        path = item['path']
        content = _raw_file(project, path, ref)
        if content is None: lerr += 1; return None
        if not content.strip(): le += 1; return None
        lf += 1
        rows = []
        for m in ANY_IMPORT_RE.finditer(content):
            stmt = m.group(0).strip()
            rows.append({**base,
                'library': _derive_library(m.group(1)),
                'import_filename': path.split('/')[-1],
                'import_file_url': f'{web_url}/-/blob/{ref}/{path}',
                'import_statement': stmt})
            ls += 1
        return rows or None
    with ThreadPoolExecutor(max_workers=FILE_WORKERS) as pool:
        for r in pool.map(process_file, src_items):
            if r: import_rows.extend(r)
    logger.info('P2 | DONE %.1fs ok=%d empty=%d err=%d imports=%d | %s',
                time.perf_counter()-t0, lf, le, lerr, len(import_rows), ns)
    _inc('source_files_fetched', lf); _inc('source_files_empty', le)
    _inc('source_files_error', lerr); _inc('import_rows_total', len(import_rows) or 1)
    if not import_rows:
        return [{**base, 'library':'', 'import_filename':'', 'import_file_url':'', 'import_statement':''}]
    return import_rows

logger.info('scan_imports defined')


## 8 · Greenplum helpers

In [ ]:
def _get_conn():
    return psycopg2.connect(
        host=DB_CONFIG['host'], port=DB_CONFIG['port'],
        dbname=DB_CONFIG['dbname'], user=DB_CONFIG['user'],
        password=DB_CONFIG['password'])

def db_test_connection():
    try:
        conn = _get_conn()
        with conn.cursor() as cur: cur.execute('SELECT 1')
        conn.close(); logger.info('DB   | connection test OK'); return True
    except Exception as e:
        logger.error('DB   | connection test FAILED: %s', e); return False

def db_setup_phase1(conn):
    col_defs = ',\n    '.join(f'"{col}" {dtype}' for col, dtype in P1_COLUMNS)
    with conn.cursor() as cur:
        cur.execute(f'DROP TABLE IF EXISTS {DB_FQTABLE};')
        cur.execute(f'CREATE TABLE {DB_FQTABLE} (\n    {col_defs}\n) DISTRIBUTED BY (project_id);')
        cur.execute(f'ALTER TABLE {DB_FQTABLE} OWNER TO erd_gpdb_prj_smart_insights;')
        cur.execute(f'GRANT SELECT ON {DB_FQTABLE} TO erd_gpdb_prj_smart_insights_ro;')
    conn.commit()
    logger.info('DB   | table created: %s', DB_FQTABLE)

def db_add_import_columns(conn):
    for col, dtype in P2_EXTRA_COLUMNS:
        try:
            with conn.cursor() as cur:
                cur.execute(f'ALTER TABLE {DB_FQTABLE} ADD COLUMN "{col}" {dtype};')
            conn.commit()
        except psycopg2.errors.DuplicateColumn:
            conn.rollback()
        except Exception as e:
            conn.rollback(); logger.warning('DB   | add col %s: %s', col, e)

def _p1_tuple(row): return tuple(_coerce(row.get(c)) for c in P1_COL_NAMES)
def _all_tuple(row): return tuple(_coerce(row.get(c)) for c in ALL_COL_NAMES)

def db_insert_p1(conn, rows):
    if not rows: logger.warning('DB | no rows'); return
    col_list = ', '.join(f'"{c}"' for c in P1_COL_NAMES)
    template = '(' + ', '.join(['%s'] * len(P1_COL_NAMES)) + ')'
    sql = f'INSERT INTO {DB_FQTABLE} ({col_list}) VALUES %s'
    tuples = [_p1_tuple(r) for r in rows]
    logger.info('DB   | INSERT %d rows', len(tuples))
    logger.info('DB   | sample[0]: %s', dict(zip(P1_COL_NAMES, tuples[0])) if tuples else None)
    inserted = 0
    with conn.cursor() as cur:
        for s in range(0, len(tuples), DB_BATCH_SIZE):
            batch = tuples[s:s+DB_BATCH_SIZE]
            psycopg2.extras.execute_values(cur, sql, batch, template=template)
            inserted += len(batch)
            logger.info('DB   | committed %d / %d', inserted, len(tuples))
    conn.commit()
    logger.info('DB   | INSERT complete -- %d rows', inserted)

def db_replace_with_imports(conn, rows):
    if not rows: return
    pids = list({_coerce(r.get('project_id')) for r in rows if r.get('project_id') is not None})
    col_list = ', '.join(f'"{c}"' for c in ALL_COL_NAMES)
    template = '(' + ', '.join(['%s'] * len(ALL_COL_NAMES)) + ')'
    tuples = [_all_tuple(r) for r in rows]
    inserted = 0
    with conn.cursor() as cur:
        cur.execute(f'DELETE FROM {DB_FQTABLE} WHERE project_id = ANY(%s)', (pids,))
        logger.info('DB   | deleted %d Phase-1 rows', cur.rowcount)
        for s in range(0, len(tuples), DB_BATCH_SIZE):
            batch = tuples[s:s+DB_BATCH_SIZE]
            psycopg2.extras.execute_values(
                cur, f'INSERT INTO {DB_FQTABLE} ({col_list}) VALUES %s', batch, template=template)
            inserted += len(batch)
            logger.info('DB   | inserted %d / %d enriched rows', inserted, len(tuples))
    conn.commit()
    logger.info('DB   | Phase-2 complete -- %d rows', inserted)

logger.info('Greenplum helpers defined')


## 9 · Test DB + fetch project list + date filter

In [ ]:
if not db_test_connection():
    raise RuntimeError('Cannot connect to Greenplum')

t_total = time.perf_counter()
logger.info("EXPORT | fetching project list for group '%s' ...", GROUP_PATH)
group     = client.groups.get(GROUP_PATH)
proj_refs = group.projects.list(include_subgroups=True, all=True, per_page=PER_PAGE)
total_all = len(proj_refs)
run_stats['total_in_group'] = total_all
logger.info('EXPORT | %d projects in group', total_all)

recent_refs = [p for p in proj_refs if is_recent(p)]
old_count   = total_all - len(recent_refs)
run_stats['recent_projects'] = len(recent_refs)
run_stats['skipped_too_old'] = old_count
logger.info('DATE   | recent=%d  too_old=%d  cutoff=%s', len(recent_refs), old_count, UPDATED_SINCE.date())
print(f'Total: {total_all}  Recent: {len(recent_refs)}  Too old: {old_count}')


## 10 · Phase 1 — parallel metadata fetch + package.json check

In [ ]:
logger.info('PHASE 1 | processing %d recent projects ...', len(recent_refs))
t_p1 = time.perf_counter()
p1_rows: List[Dict[str, Any]] = []
p1_done = 0
p1_total = len(recent_refs)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    fmap = {pool.submit(fetch_project_with_pkg, ref): ref for ref in recent_refs}
    for future in as_completed(fmap):
        p1_done += 1
        ref = fmap[future]
        try: row = future.result(timeout=FUTURE_TIMEOUT)
        except FutureTimeout:
            logger.warning('P1 | TIMEOUT | %s', ref.path_with_namespace)
            _inc('p1_timeouts'); row = None
        except Exception as e:
            logger.error('P1 | ERROR | %s | %s', ref.path_with_namespace, e)
            _inc('p1_errors'); row = None
        if row: p1_rows.append(row)
        if p1_done % PROGRESS_EVERY == 0 or p1_done == p1_total:
            pct = 100 * p1_done // p1_total
            print(f'\r  P1: {p1_done}/{p1_total} ({pct}%) '
                  f'| has_pkg={len(p1_rows)} '
                  f'no_pkg={run_stats["skipped_no_package_json"]} '
                  f'err={run_stats["p1_errors"]} timeout={run_stats["p1_timeouts"]}   ',
                  end='', flush=True)
print()
t_p1_elapsed = time.perf_counter() - t_p1
logger.info('PHASE 1 | done %.1fs | rows=%d', t_p1_elapsed, len(p1_rows))


## 11 · Insert Phase-1 rows into Greenplum

In [ ]:
print(f'Inserting {len(p1_rows)} rows into Greenplum ...')
conn = _get_conn()
db_setup_phase1(conn)
db_insert_p1(conn, p1_rows)
conn.close()
print(f'OK  {len(p1_rows)} rows written to {DB_FQTABLE}')


## 12 · Phase 1 summary + user gate

In [ ]:
print()
print('=' * 70)
print('  PHASE 1 COMPLETE')
print('=' * 70)
print(f'  Total projects in group             : {total_all}')
print(f'  Updated since {UPDATED_SINCE.date()}           : {len(recent_refs)}')
print(f'    -- too old (skipped)              : {old_count}')
print(f'  Have package.json  -> in DB         : {len(p1_rows)}')
print(f'    -- no package.json (skipped)      : {run_stats["skipped_no_package_json"]}')
print(f'  Phase 1 elapsed                     : {t_p1_elapsed:.1f}s')
print('=' * 70)
print()
_proceed = input(f'Proceed to import-statement scan for {len(p1_rows)} projects? (yes/no): ').strip().lower()
if _proceed not in ('yes', 'y'):
    t_elapsed = time.perf_counter() - t_total
    print(f'Stopped. Data is in {DB_FQTABLE}. Total elapsed: {t_elapsed:.1f}s')
    raise SystemExit('Stopped after Phase 1')
logger.info('USER | proceeding to Phase 2')


## 13 · Phase 2 — parallel import scan

In [ ]:
logger.info('PHASE 2 | scanning %d projects ...', len(p1_rows))
t_p2 = time.perf_counter()
p2_rows: List[Dict[str, Any]] = []
p2_done = 0
p2_total = len(p1_rows)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    fmap = {pool.submit(scan_imports, row): row for row in p1_rows}
    for future in as_completed(fmap):
        p2_done += 1
        row = fmap[future]
        ns  = row.get('path_with_namespace', '?')
        try: result_rows = future.result(timeout=FUTURE_TIMEOUT)
        except FutureTimeout:
            logger.warning('P2 | TIMEOUT | %s', ns)
            _inc('p2_timeouts'); result_rows = None
        except Exception as e:
            logger.error('P2 | ERROR | %s | %s', ns, e)
            _inc('p2_errors'); result_rows = None
        if result_rows:
            p2_rows.extend(result_rows)
            logger.info('P2 | %d/%d +%d rows | %s', p2_done, p2_total, len(result_rows), ns)
        if p2_done % PROGRESS_EVERY == 0 or p2_done == p2_total:
            pct = 100 * p2_done // p2_total
            print(f'\r  P2: {p2_done}/{p2_total} ({pct}%) '
                  f'| rows={len(p2_rows)} imports={run_stats["import_rows_total"]} '
                  f'err={run_stats["p2_errors"]} timeout={run_stats["p2_timeouts"]}   ',
                  end='', flush=True)
print()
t_p2_elapsed = time.perf_counter() - t_p2
logger.info('PHASE 2 | done %.1fs | %d rows', t_p2_elapsed, len(p2_rows))


## 14 · Write Phase-2 rows to Greenplum

In [ ]:
print(f'Writing {len(p2_rows)} enriched rows to Greenplum ...')
conn = _get_conn()
db_add_import_columns(conn)
db_replace_with_imports(conn, p2_rows)
conn.close()
print(f'OK  {len(p2_rows)} rows written to {DB_FQTABLE}')


## 15 · Final summary

In [ ]:
t_elapsed = time.perf_counter() - t_total
print()
print('=' * 70)
print('  FINAL SUMMARY')
print('=' * 70)
print(f'  Total projects in group  : {total_all}')
print(f'  Have package.json        : {len(p1_rows)}')
print(f'  No package.json          : {run_stats["skipped_no_package_json"]}')
print(f'  Import rows              : {len(p2_rows)}')
print(f'  Phase 1 elapsed (s)      : {t_p1_elapsed:.1f}')
print(f'  Phase 2 elapsed (s)      : {t_p2_elapsed:.1f}')
print(f'  Total elapsed (s)        : {t_elapsed:.1f}')
print('=' * 70)
